## **Binary Data-preprocessing**


In [ ]:
import os
import pandas as pd

from config import (
    CSV_PATH,
    XRAY_DIR,
    OUT_DIR,
)


def prepare_binary_dataset(csv_path: str, image_folder: str) -> pd.DataFrame:
    # Load and rename columns for consistency
    df = pd.read_csv(csv_path)
    df = df.rename(
        columns={
            "Image Index": "IMGPATH",
            "Finding Labels": "DL",
            "Patient ID": "ID",
            "Patient Age": "AGE",
            "Patient Gender": "GENDER",
            "View Position": "VP",
        }
    )

    # Define original diseases to classify as "Disease Present"
    disease_list = {
        "Atelectasis",
        "Cardiomegaly",
        "Effusion",
        "Infiltration",
        "Mass",
        "Nodule",
        "Pneumonia",
        "Pneumothorax",
    }

    # Map image paths to their presence in the folder
    image_paths = {
        os.path.basename(file): os.path.join(root, file)
        for root, _, files in os.walk(image_folder)
        for file in files
    }
    df = df[df["IMGPATH"].isin(image_paths)].copy()
    df["IMGPATH"] = df["IMGPATH"].apply(lambda x: image_paths.get(x, x))

    # Filter dataset to only include relevant diseases and "No Finding"
    df = df[
        df["DL"].apply(
            lambda x: x == "No Finding" or any(d in x.split("|") for d in disease_list)
        )
    ]

    # Assign binary labels (1: Disease Present, 0: No Finding)
    df["HOTLABEL"] = df["DL"].apply(
        lambda x: 1 if any(d in x.split("|") for d in disease_list) else 0
    )

    # Sort by Patient ID and Follow-up # to get the latest image
    df = df.sort_values(by=["ID", "Follow-up #"], ascending=[True, False])

    # Identify patients who only have "No Finding"
    patient_has_disease = df.groupby("ID")["HOTLABEL"].max()
    df = df.merge(patient_has_disease, on="ID", suffixes=("", "_MAX"))

    # Keep only the latest disease image for patients with disease
    df_disease = df[(df["HOTLABEL_MAX"] == 1) & (df["HOTLABEL"] == 1)].drop_duplicates(
        subset=["ID"], keep="first"
    )

    # Keep the latest "No Finding" image for patients who never had disease
    df_no_finding = df[
        (df["HOTLABEL_MAX"] == 0) & (df["HOTLABEL"] == 0)
    ].drop_duplicates(subset=["ID"], keep="first")

    # Balance dataset by matching "No Finding" cases to "Disease Present" cases
    neg_samples = (
        df_no_finding.sample(n=len(df_disease), random_state=42, replace=False)
        if len(df_no_finding) > len(df_disease)
        else df_no_finding
    )

    # Combine balanced dataset
    final_df = pd.concat([df_disease, neg_samples]).reset_index(drop=True)

    # Keep only relevant columns in output
    final_df = final_df[["IMGPATH", "DL", "ID", "AGE", "GENDER", "VP", "HOTLABEL"]]

    return final_df


final_df: pd.DataFrame = prepare_binary_dataset(CSV_PATH, XRAY_DIR)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split


def split_dataset(
    df: pd.DataFrame,
    bayesian_calib_size: int = 1500,
    test_size: float = 0.15,
    val_size: float = 0.15,
    random_state: int = 42,
):
    # Split train + temp (validation + Bayesian calibration + test)
    train_df, temp_df = train_test_split(
        df,
        test_size=val_size + test_size,
        stratify=df["HOTLABEL"],
        random_state=random_state,
    )

    # Further split temp into validation + test
    val_df, test_df = train_test_split(
        temp_df,
        test_size=test_size / (val_size + test_size),
        stratify=temp_df["HOTLABEL"],
        random_state=random_state,
    )

    # Select Bayesian calibration set randomly from validation set
    calib_df = val_df.sample(
        n=bayesian_calib_size, random_state=random_state, replace=False
    )

    # Remove Bayesian calibration samples from validation set
    val_df = val_df.drop(calib_df.index)

    # Save datasets to OUT_DIR
    os.makedirs(OUT_DIR, exist_ok=True)
    train_df.to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)
    val_df.to_csv(os.path.join(OUT_DIR, "validation.csv"), index=False)
    test_df.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)
    calib_df.to_csv(os.path.join(OUT_DIR, "inferno.csv"), index=False)

    return {
        "train": train_df,
        "validation": val_df,
        "test": test_df,
        "inferno": calib_df,
    }


split_data = split_dataset(final_df)
train_set, val_set, test_set, calib_set = (
    split_data["train"],
    split_data["validation"],
    split_data["test"],
    split_data["inferno"],
)